In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import * 
from delta.tables import DeltaTable

In [0]:
order_detail = spark.read.format("parquet").load("abfss://bronze@intechaccstorage.dfs.core.windows.net/SalesOrderDetail")
order_detail.limit(5).display()

SalesOrderID,SalesOrderDetailID,OrderQty,ProductID,UnitPrice,UnitPriceDiscount,LineTotal,rowguid,ModifiedDate,_rescued_data
71774,110562,1,836,356.8980,0.0000,356.898000,e3a1994c-7a68-4ce8-96a3-77fdd3bbd730,2008-06-01 00:00:00.0000000,null
71774,110563,1,822,356.8980,0.0000,356.898000,5c77f557-fdb6-43ba-90b9-9a7aec55ca32,2008-06-01 00:00:00.0000000,null
71776,110567,1,907,63.9000,0.0000,63.900000,6dbfe398-d15d-425e-aa58-88178fe360e5,2008-06-01 00:00:00.0000000,null
71780,110616,4,905,218.4540,0.0000,873.816000,377246c9-4483-48ed-a5b9-e56f005364e0,2008-06-01 00:00:00.0000000,null
71780,110617,2,983,461.6940,0.0000,923.388000,43a54bcd-536d-4a1b-8e69-24d083507a14,2008-06-01 00:00:00.0000000,null


In [0]:
order_detail = order_detail.withColumn('OrderQty', col("OrderQty").cast("float"))\
                           .withColumn('UnitPrice', col("UnitPrice").cast("float"))\
                           .withColumn('UnitPriceDiscount', col("UnitPriceDiscount").cast("float"))\
                           .withColumn('LineTotal', col("LineTotal").cast("float"))\
                           .withColumn('ModifiedDate', to_date(col("ModifiedDate")))
                                       
order_detail = order_detail.withColumnRenamed("SalesOrderID", "Sales_order_id")\
                           .withColumnRenamed("SalesOrderDetailID", "Sales_order_detail_id")\
                           .withColumnRenamed("OrderQty", "Order_qty")\
                           .withColumnRenamed("ProductID", "Product_id")\
                           .withColumnRenamed("UnitPrice", "Unit_price")\
                           .withColumnRenamed("UnitPriceDiscount", "Unit_price_discount")\
                           .withColumnRenamed("LineTotal", "Line_total")\
                           .withColumnRenamed("ModifiedDate", "Modified_date")


In [0]:
order_detail = order_detail.drop("rowguid", "_rescued_data")
order_detail.columns

['Sales_order_id',
 'Sales_order_detail_id',
 'Order_qty',
 'Product_id',
 'Unit_price',
 'Unit_price_discount',
 'Line_total',
 'Modified_date']

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricksintech.silver.order_detail
(
    Sales_order_id STRING,
    Sales_order_detail_id STRING,
    Order_qty FLOAT,
    Product_id STRING,
    Unit_price FLOAT,
    Unit_price_discount FLOAT,
    Line_total FLOAT,
    Modified_date DATE
) 
USING DELTA
LOCATION 'abfss://silver@intechaccstorage.dfs.core.windows.net/order_detail';

In [0]:
# Reference target Delta table
silver_table = DeltaTable.forName(spark, "databricksintech.silver.order_detail")

# Execute the Merge (Upsert)
silver_table.alias("target").merge(
    source = order_detail.alias("source"),
    condition = (
        "target.Sales_order_detail_id = source.Sales_order_detail_id"
        
    )
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Done!")

Done!
